[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/sh2026-workshop/workshop/02_rules_and_gazetteers.ipynb)

# AI and NLP for Spatial Humanities
## 02 - Rules and gazetteers: transparent, reproducible, bounded

**Duration:** about 45 minutes

We now apply a deliberately transparent deterministic baseline to the same kinds of evidence used in Notebook 01.

The aim is not to build the cleverest possible rule system. It is to ask:

> **What do we gain from a method whose vocabulary and decisions we can inspect completely, and what does it fail to see when the text moves outside those assumptions?**

### Learning outcomes
By the end you should be able to:
1. distinguish rule/resource matching from contextual NER;
2. inspect exactly what a bounded gazetteer contains;
3. measure rule coverage against the human reference;
4. reproduce failures caused by morphology, spelling variation and unseen names;
5. separate toponym recognition from geographic resolution;
6. explain why transparent rules can still encode historical/theoretical assumptions.

In [1]:
!wget -q https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/sh2026-workshop/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

# Bound from the shared context: the cells below were written against these.
repo_dir = ctx.repo
data_dir = ctx.data

import json, time
FAST_MODE=True
print("Repository:",repo_dir)


Reusing existing checkout at /home/ezeani/workspace/spatial-humanities-2026
Dependencies already installed in this runtime.

Ready in 0s.
  repo    : /home/ezeani/workspace/spatial-humanities-2026
  commit  : 798f2be
  data    : /home/ezeani/workspace/spatial-humanities-2026/workshop/data
  outputs : /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs
  route   : CPU only, no API key needed

If this cell failed, put your hand up. Do not re-run it more than once.
Repository: /home/ezeani/workspace/spatial-humanities-2026


In [2]:
import pandas as pd
from IPython.display import display
from spatio_textual.gold import assert_valid_gold, load_gold_jsonl, score_span_annotations
from spatio_textual.rules import RuleGazetteerAnnotator, filter_supported_gold_labels, load_teaching_gazetteer

gold_path=repo_dir/"workshop"/"data"/"gold_reference_v0.1.jsonl"
records=load_gold_jsonl(gold_path); assert_valid_gold(records)
gold={r["example_id"]:r for r in records}
gazetteer_path=repo_dir/"workshop"/"data"/"teaching_gazetteer.csv"
gazetteer=load_teaching_gazetteer(gazetteer_path)
display(pd.DataFrame(gazetteer))

,text,label,conceptual_level,note
0,Penrith,TOPONYM,location,Teaching gazetteer entry for the Lake District...
1,Pooley Bridge,TOPONYM,location,Teaching gazetteer entry for the Lake District...
2,Eamont,TOPONYM,location,Hydrographic name retained as written
3,Ulleswater,TOPONYM,location,Historical or variant source spelling must be ...
4,Amsterdam,TOPONYM,location,Synthetic oral-history teaching example
5,Auschwitz,TOPONYM,location,Synthetic oral-history teaching example; textu...
6,London,TOPONYM,location,Synthetic teaching examples
7,Cambridge,TOPONYM,location,Deliberately ambiguous place name; gazetteer r...
8,Czechoslovakia,TOPONYM,location,Historical polity; do not silently normalize t...


## 1. Important benchmark warning

The CSV above is a **teaching gazetteer**. It deliberately contains names from the visible exercises.

That makes it useful for showing how deterministic matching works, but it also means performance on those same names is **not evidence of generalisation**.

For the final keynote benchmark we will freeze a held-out set before final rule/prompt tuning.

In [3]:
ann=RuleGazetteerAnnotator(
    gazetteer_path=gazetteer_path,
    include_project_resources=True,
    case_sensitive=False,
    link_places=False,
)
print("Loaded deterministic patterns:",ann.pattern_count)

/home/ezeani/workspace/spatial-humanities-2026/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loaded deterministic patterns: 80


## 2. Run the bounded rule baseline on the Lake District passage

The baseline combines:
- project resource lists such as geo-nouns;
- the bounded teaching gazetteer;
- a few explicit phrase/regex rules for distance, direction, movement, time and transport.

There is no statistical NER model and no LLM in this step.

In [4]:
record=gold["cldw_penrith_pooley_bridge"]
text=record["text"]
result=ann.annotate(text)
rule_spans=result["spans"]
display(pd.DataFrame(rule_spans))
display(pd.DataFrame(result["telemetry"]))

,text,label,start_char,end_char,source,confidence
0,Penrith,TOPONYM,5,12,rule:entity_ruler,1.0
1,Pooley Bridge,TOPONYM,31,44,rule:entity_ruler,1.0
2,about six miles distant,DISTANCE,46,69,rule:distance_regex,1.0
3,Eamont,TOPONYM,87,93,rule:entity_ruler,1.0
4,Ulleswater,TOPONYM,117,127,rule:entity_ruler,1.0


,task,backend,provider,model,latency_ms,input_chars,input_tokens_est,output_tokens_est,cost_usd_est,success,error
0,rule_gazetteer_spatial_annotation,rules,local,entity_ruler+regex,1.409,128,32,163,0.0,True,None


In [5]:
reference=filter_supported_gold_labels(record["spans"])
score_exact=score_span_annotations(rule_spans,reference,match="exact",label_sensitive=True)
score_overlap=score_span_annotations(rule_spans,reference,match="overlap",label_sensitive=True)
display(pd.DataFrame([
    {k:v for k,v in score_exact.items() if k in {"match","tp","fp","fn","precision","recall","f1"}},
    {k:v for k,v in score_overlap.items() if k in {"match","tp","fp","fn","precision","recall","f1"}},
]))
print("Reference items not recovered exactly:")
display(pd.DataFrame([reference[i] for i in score_exact["unmatched_ref_indices"]]))

,match,tp,fp,fn,precision,recall,f1
0,exact,5,0,1,1.0,0.833333,0.909091
1,overlap,5,0,1,1.0,0.833333,0.909091


Reference items not recovered exactly:


,span_id,layer,label,text,start_char,end_char,conceptual_level,certainty,attributes,notes
0,s002,entity,GEONOUN,roads,17,22,locale,explicit,{'subtype': 'route_feature'},None


### Read the failure, not only the score

The project geo-noun resource currently contains **`road`**, not **`roads`**. The rule system therefore has a highly interpretable failure on the plural form.

That is valuable evidence: the error can be traced to a concrete resource decision rather than an opaque model state.

In [6]:
tests=[
    "A road crossed the village.",
    "Two roads crossed the village.",
    "We stayed near the river.",
]
for t in tests:
    spans=ann.annotate(t)["spans"]
    print("\nTEXT:",t)
    print([(s["text"],s["label"],s["source"]) for s in spans])


TEXT: A road crossed the village.
[('road', 'GEONOUN', 'rule:entity_ruler'), ('crossed', 'MOVEMENT_CUE', 'rule:entity_ruler'), ('village', 'GEONOUN', 'rule:entity_ruler')]

TEXT: Two roads crossed the village.
[('crossed', 'MOVEMENT_CUE', 'rule:entity_ruler'), ('village', 'GEONOUN', 'rule:entity_ruler')]

TEXT: We stayed near the river.
[('near', 'SPATIAL_RELATION', 'rule:entity_ruler'), ('river', 'GEONOUN', 'rule:entity_ruler')]


## 3. Controlled perturbation experiment

Rules can be robust to some changes and brittle to others.

The default teaching matcher is case-insensitive, so capitalization changes should not matter. But spelling variation and names absent from the gazetteer should.

In [7]:
perturbations=pd.DataFrame([
    ("known exact","We left Penrith for London."),
    ("case change","We left PENRITH for LONDON."),
    ("spelling variation","We left Penrithh for London."),
    ("unseen toponym","We left Keswick for London."),
],columns=["condition","text"])

rows=[]
for _,row in perturbations.iterrows():
    out=ann.annotate(row["text"])
    rows.append({
        "condition":row["condition"],
        "text":row["text"],
        "matches":[(s["text"],s["label"]) for s in out["spans"]],
        "latency_ms":out["telemetry"][0]["latency_ms"],
    })
display(pd.DataFrame(rows))

,condition,text,matches,latency_ms
0,known exact,We left Penrith for London.,"[(left, MOVEMENT_CUE), (Penrith, TOPONYM), (Lo...",0.244
1,case change,We left PENRITH for LONDON.,"[(left, MOVEMENT_CUE), (PENRITH, TOPONYM), (LO...",0.216
2,spelling variation,We left Penrithh for London.,"[(left, MOVEMENT_CUE), (London, TOPONYM)]",0.139
3,unseen toponym,We left Keswick for London.,"[(left, MOVEMENT_CUE), (London, TOPONYM)]",0.120


### Interpretation

This is a central advantage and limitation of rules:

- failure modes are often easy to explain;
- behaviour is reproducible;
- extending coverage can be straightforward;
- but every new spelling, morphology, historical form or relation pattern creates maintenance work;
- bounded success can be mistaken for generality if evaluation uses the same gazetteer used to build the system.

## 4. Recognition is not resolution

Finding the string **Cambridge** is one task. Deciding which Cambridge it denotes is another.

Now enable the offline resolver and inspect what happens.

In [8]:
linked=RuleGazetteerAnnotator(gazetteer_path=gazetteer_path,link_places=True)
cam=linked.annotate(gold["synthetic_ambiguous_cambridge"]["text"])
display(pd.DataFrame(cam["spans"]))
print("Requires review:",cam["requires_review"])
print("Review notes:",cam["review_notes"])

,text,label,start_char,end_char,source,confidence,resolved_name,lat,lon,place_type_resolved,resolution_status,geo_source,geo_confidence,ambiguous,candidates_count,candidates,geonameid,countrycode
0,left,MOVEMENT_CUE,19,23,rule:entity_ruler,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Cambridge,TOPONYM,24,33,rule:entity_ruler,1.0,Cambridge,52.20000,0.11667,CITY,resolved_ambiguous,geonamescache:city,0.62,True,4.0,"[{'name': 'Cambridge', 'countrycode': 'GB', 'p...",2653941.0,GB
2,travelled,MOVEMENT_CUE,38,47,rule:entity_ruler,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,London,TOPONYM,51,57,rule:entity_ruler,1.0,London,51.50853,-0.12574,CITY,resolved_ambiguous,geonamescache:city,0.62,True,2.0,"[{'name': 'London', 'countrycode': 'GB', 'popu...",2643743.0,GB


Requires review: True
Review notes: ['One or more linked places are ambiguous or unresolved.']


A deterministic recognizer can be completely certain that the token string `Cambridge` matched its gazetteer while the **geographic resolution remains ambiguous**. Those uncertainties must not be collapsed into one confidence number.

## 5. Historical geography: prefer unresolved to anachronistically wrong

The lightweight resolver previously treated `Czechoslovakia` as an alias of a present-day country. For SH2026 we changed that behaviour.

Because this offline gazetteer is not time-indexed, the safer default is now:
- preserve `Czechoslovakia` exactly;
- mark it `HISTORICAL_POLITY`;
- leave coordinates unresolved;
- require review.

This is a methodological decision, not just a software bug fix.

In [9]:
hist=linked.annotate(gold["synthetic_historical_polity"]["text"])
hist_rows=[s for s in hist["spans"] if s["text"]=="Czechoslovakia"]
display(pd.DataFrame(hist_rows))

,text,label,start_char,end_char,source,confidence,resolved_name,lat,lon,place_type_resolved,resolution_status,geo_source,geo_confidence,ambiguous,candidates_count,candidates,historical_name,review_reason
0,Czechoslovakia,TOPONYM,28,42,rule:entity_ruler,1.0,Czechoslovakia,None,None,HISTORICAL_POLITY,unresolved,historical_name:preserved,0.0,True,0,[],True,Historical polity requires time-aware/human re...


## 6. Rules can encode theory too

Transparent does not mean neutral.

A rule inventory decides:
- which nouns count as geographical;
- which spellings are recognized;
- which movement verbs matter;
- which historical names are normalized;
- which relation words are considered spatial.

The advantage is that we can inspect those assumptions directly. The challenge is that they still need scholarly justification.

## 7. Build a small rule-baseline comparison table

We run the same deterministic baseline across the public-safe teaching examples and save the results for later notebooks/keynote figures.

In [10]:
comparison=[]
for rec in records:
    out=ann.annotate(rec["text"])
    ref=filter_supported_gold_labels(rec["spans"])
    score=score_span_annotations(out["spans"],ref,match="exact",label_sensitive=True)
    comparison.append({
        "example_id":rec["example_id"],
        "method":"rules",
        "backend":"entity_ruler+regex",
        "reference_spans":len(ref),
        "predicted_spans":len(out["spans"]),
        "precision":score["precision"],
        "recall":score["recall"],
        "f1":score["f1"],
        "latency_ms":out["telemetry"][0]["latency_ms"],
        "cost_usd_est":0.0,
        "unmatched_reference":len(score["unmatched_ref_indices"]),
    })
comparison_df=pd.DataFrame(comparison)
display(comparison_df)

out_dir=repo_dir/"sh2026_outputs"/"comparisons"
out_dir.mkdir(parents=True,exist_ok=True)
csv_path=out_dir/"rules_baseline_teaching_reference.csv"
comparison_df.to_csv(csv_path,index=False)
print("Saved:",csv_path)

,example_id,method,backend,reference_spans,predicted_spans,precision,recall,f1,latency_ms,cost_usd_est,unmatched_reference
0,cldw_penrith_pooley_bridge,rules,entity_ruler+regex,6,5,1.000000,0.833333,0.909091,0.239,0.0,1
1,synthetic_qa_journey,rules,entity_ruler+regex,8,10,0.500000,0.625000,0.555556,0.614,0.0,3
2,synthetic_ambiguous_cambridge,rules,entity_ruler+regex,5,4,1.000000,0.800000,0.888889,0.231,0.0,1
3,synthetic_historical_polity,rules,entity_ruler+regex,5,3,0.666667,0.400000,0.500000,0.314,0.0,3
4,synthetic_relational_space,rules,entity_ruler+regex,11,7,0.857143,0.545455,0.666667,1.390,0.0,5


Saved: /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs/comparisons/rules_baseline_teaching_reference.csv


## 8. What this baseline can and cannot support

**Strong claims we can make:**
- the method is deterministic for fixed resources/configuration;
- its pattern inventory is inspectable;
- local inference has no per-call API charge;
- errors such as the `road`/`roads` miss are directly traceable;
- historical/ambiguous resolution can be explicitly surfaced.

**Claims we should not make from this exercise:**
- that the teaching gazetteer generalises to unseen corpora;
- that a gazetteer match proves the correct real-world referent;
- that deterministic equals unbiased;
- that failure on an unseen form proves rules are intrinsically inferior;
- that performance on this visible set is the final keynote benchmark.

### Core message

> **Transparent and reproducible does not mean complete. But incompleteness that we can inspect can be methodologically valuable.**

Next: **03 - Contextual NER**, where the system gains contextual generalisation but also imports training-data and model-ontology assumptions that are harder to inspect directly.